# Standalone Imaging Center Viability

  The client is a rural hospital considering opening a standalone imaging center in an adjacent community. It uses synthetic data for a visits table, charge master and posted charges. The data is cleaned and validated using DuckDB SQL in Jupyter Notebook. While the project is based on a real project I did for a client, all patient and charge information is synthetic and the zip codes used are not the same as the client facility for whom I did the project.

**Tools:** Jupyter Notebook, DuckDB SQL, Python (DuckDB package)  
**Data Source:** Synthetic data created for portfolio demonstration; no protected health information is included.

## Project Objectives

 - Standardize field formats and data types across source CSV files  
 - Parse mixed date formats into analysis-ready date fields  
 - Validate data quality  
 - Export cleaned CSV files for downstream analysis  

In [1]:
import duckdb

## Source Files and Data Limitations

   The project uses synthetic CSV files stored in the 'data/raw' directory. The data is intended only to demonstrate data cleaning, validation, and SQL analysis techniques. It does not represent real patients, visits or charges. 

## Raw Data Profile

In [2]:
duckdb.sql("""
    SELECT
        'visits' AS dataset,
        COUNT(*) AS row_count
    FROM 'Data Sets/Imaging Center Data/visits.csv' 

    UNION ALL

    SELECT
        'charges' AS dataset,
        COUNT(*) AS row_count
    FROM 'Data Sets/Imaging Center Data/charges.csv' 

    UNION ALL

    SELECT
        'chgMast' AS dataset,
        COUNT(*) AS row_count
    FROM 'Data Sets/Imaging Center Data/chgMast.csv';
""")

┌─────────┬───────────┐
│ dataset │ row_count │
│ varchar │   int64   │
├─────────┼───────────┤
│ visits  │      1000 │
│ charges │      1217 │
│ chgMast │      2500 │
└─────────┴───────────┘

## Clean Charge Master

   The charge master has charge descriptions in mixed case. All upper case would make text searches cleaner. Using UPPER() to convert everything. The charge amount is not standardized with 2 decimal places. Casting the amount as DECIMAL() with a specified 2 decimal places.

In [3]:
duckdb.sql("""
COPY(
SELECT
    chgNo as "chgNo",
    UPPER(description) as "description",
    CPT as "CPT",
    revCode as "revCode",
    CAST(chgAmt AS DECIMAL(10, 2)) as "chgAmt"
FROM
    'Data Sets/Imaging Center Data/chgMast.csv'
)
    TO 'Data Sets/Imaging Center Data/chgMast_Cleaned_SQL.csv'
    WITH (HEADER, DELIMITER ',');
""")

## Charge Master Data Validation

In [4]:
duckdb.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(chgNo) AS rows_with_charge_numbers,
    (COUNT(*) - COUNT(chgNo)) AS missing_charge_number_rows
FROM 'Data Sets/Imaging Center Data/chgMast_Cleaned_SQL.csv';
""")

┌────────────┬──────────────────────────┬────────────────────────────┐
│ total_rows │ rows_with_charge_numbers │ missing_charge_number_rows │
│   int64    │          int64           │           int64            │
├────────────┼──────────────────────────┼────────────────────────────┤
│       2500 │                     2500 │                          0 │
└────────────┴──────────────────────────┴────────────────────────────┘

## Clean Patient Visits

   The dates in the table have mixed formats. Using try_strptime() to clean and standardize the dates.

In [5]:
duckdb.sql("""
COPY(
SELECT
    acctNum,
    mrn,
    CAST(
        try_strptime(
            TRIM(DOB),
            ['%m/%d/%Y', '%Y-%m-%d']
       )
       AS DATE
       ) AS "DOB",
    sex,
    CAST(
        try_strptime(
            TRIM(admitDate),
            ['%m/%d/%Y', '%Y-%m-%d']
       )
       AS DATE
       ) AS "admitDate",
    CAST(
        try_strptime(
            TRIM(dischDate),
            ['%m/%d/%Y', '%Y-%m-%d']
       )
       AS DATE
       ) AS "dischDate",
    zip
FROM
    'Data Sets/Imaging Center Data/visits.csv'
)  
TO 'Data Sets/Imaging Center Data/visits_cleaned_SQL.csv'
WITH (HEADER, DELIMITER ',');
""")

## Patient Visits Data Validation

In [6]:
duckdb.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(acctNum) AS rows_with_account_numbers,
    (COUNT(*) - COUNT(acctNum)) AS missing_account_number_rows
FROM 'Data Sets/Imaging Center Data/visits_cleaned_SQL.csv';
""")

┌────────────┬───────────────────────────┬─────────────────────────────┐
│ total_rows │ rows_with_account_numbers │ missing_account_number_rows │
│   int64    │           int64           │            int64            │
├────────────┼───────────────────────────┼─────────────────────────────┤
│       1000 │                      1000 │                           0 │
└────────────┴───────────────────────────┴─────────────────────────────┘

## Clean Charges

   The service dates have mixed formats. Using try_strptime() to standardized the dates.

In [7]:
duckdb.sql("""
COPY(
    SELECT
       acctNum,
       chgNo,
       qty,
       CAST(
           try_strptime(
               TRIM(svcDate),
               ['%m/%d/%Y', '%Y-%m-%d']
           )
           AS DATE
        )AS svcDate
    FROM 'Data Sets/Imaging Center Data/charges.csv'
    )
TO 'Data Sets/Imaging Center Data/charges_cleaned_SQL.csv'
WITH (HEADER, DELIMITER ',');
""")

## Patient Charges Validation

In [8]:
duckdb.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(acctNum) AS rows_with_account_numbers,
    (COUNT(*) - COUNT(acctNum)) AS missing_account_number_rows
FROM 'Data Sets/Imaging Center Data/charges_cleaned_SQL.csv';
""")

┌────────────┬───────────────────────────┬─────────────────────────────┐
│ total_rows │ rows_with_account_numbers │ missing_account_number_rows │
│   int64    │           int64           │            int64            │
├────────────┼───────────────────────────┼─────────────────────────────┤
│       1217 │                      1217 │                           0 │
└────────────┴───────────────────────────┴─────────────────────────────┘

## Cross-File Validation

   This section verifies that cleaned records can be linked through their key identifiers.

In [9]:
duckdb.sql("""
    SELECT
        v.acctNum AS "Account #",
        v.mrn AS "Medical Record #",
        v.DOB AS "Date of Birth",
        v.sex as "Gender",
        v.admitDate as "Admit Date",
        v.dischDate as "Discharge Date",
        v.zip AS "Patient Zip Code",
        c.svcDate AS "Date of Service",
        c.chgNo AS "Charge #",
        m.description AS "Charge Description",
        m.CPT AS "CPT Code",
        m.revCode AS "Revenue Code",
        c.qty AS "Quantity",
        (c.qty * m.chgAmt) AS "Charge Amt"
    FROM 'Data Sets/Imaging Center Data/charges_cleaned_SQL.csv' as c
    JOIN 'Data Sets/Imaging Center Data/chgMast_cleaned_SQL.csv' as m
        ON c.chgNo = m.chgNo
    JOIN 'Data Sets/Imaging Center Data/visits_cleaned_SQL.csv' as v
        ON c.acctNum = v.acctNum
    LIMIT 25
""")

┌───────────┬──────────────────┬───────────────┬─────────┬────────────┬────────────────┬──────────────────┬─────────────────┬──────────┬───────────────────────────────────────┬──────────┬──────────────┬──────────┬────────────┐
│ Account # │ Medical Record # │ Date of Birth │ Gender  │ Admit Date │ Discharge Date │ Patient Zip Code │ Date of Service │ Charge # │          Charge Description           │ CPT Code │ Revenue Code │ Quantity │ Charge Amt │
│   int64   │      int64       │     date      │ varchar │    date    │      date      │      int64       │      date       │  int64   │                varchar                │  int64   │   varchar    │  int64   │   double   │
├───────────┼──────────────────┼───────────────┼─────────┼────────────┼────────────────┼──────────────────┼─────────────────┼──────────┼───────────────────────────────────────┼──────────┼──────────────┼──────────┼────────────┤
│ 900000973 │         50000088 │ 1954-11-09    │ F       │ 2025-01-10 │ 2025-01-10     │    

## Analysis Example 1

   The purpose of this first analysis is to show the number of studies by modality along with the revenue generated from each for the target zip code where the proposed imaging center would be built.

In [10]:
duckdb.sql("""
    SELECT
        CASE
            WHEN m.revCode IN(0320, 0321, 0322, 0323, 0324, 0329) THEN 'Diagnostic X-Rays'
            WHEN m.revCode IN(0401, 0403) THEN 'Mammography'
            WHEN m.revCode IN(0402) THEN 'Ultrasound'
            WHEN m.revCode IN(0350, 0351, 0352, 0359) THEN 'CT'
            WHEN m.revCode IN(0610, 0611, 0612, 0614, 0615, 0616) THEN 'MRA/MRI'
            ELSE m.revCode
        END AS "Modality",
        SUM(c.qty) as "Exams/Modality",
        CAST(SUM((c.qty * m.chgAmt)) AS DECIMAL(12,2)) as "Revenue/Modality"
    FROM 'Data Sets/Imaging Center Data/charges_cleaned_SQL.csv' as c
    JOIN 'Data Sets/Imaging Center Data/chgMast_cleaned_SQL.csv' as m
        ON c.chgNo = m.chgNo
    JOIN 'Data Sets/Imaging Center Data/visits_cleaned_SQL.csv' as v
        ON c.acctNum = v.acctNum
    WHERE v.zip = 30547
    GROUP BY "Modality"
""")

┌───────────────────┬────────────────┬──────────────────┐
│     Modality      │ Exams/Modality │ Revenue/Modality │
│      varchar      │     int128     │  decimal(12,2)   │
├───────────────────┼────────────────┼──────────────────┤
│ Ultrasound        │             51 │         16082.00 │
│ Mammography       │             37 │          9992.75 │
│ MRA/MRI           │             41 │         82071.25 │
│ CT                │             55 │         85239.50 │
│ Diagnostic X-Rays │             95 │         20823.75 │
└───────────────────┴────────────────┴──────────────────┘

## Analysis Example 2

   This example further examines the data to determine the age ranges of the patients seen. The ages, specifically those 65 and older, could indicate Medicare reimbursement rates as opposed to commercial payers.

In [11]:
duckdb.sql("""
WITH patientAge AS(
    SELECT 
        v.mrn,
        date_sub('year', v.DOB, CURRENT_DATE) AS AGE
    FROM 'Data Sets/Imaging Center Data/visits_cleaned_SQL.csv' AS v
    WHERE v.ZIP = 30547)
    SELECT
        CASE
            WHEN patientAge.age <= 18 THEN 'Pediatric'
            WHEN patientAge.age >18 AND patientAge.age < 65 THEN 'Adult'
            WHEN patientAge.age >= 65 then 'Senior'
            ELSE CAST(patientAge.age AS VARCHAR)
        END AS "Age Group",
        COUNT(DISTINCT patientAge.mrn)
    FROM patientAge
    GROUP BY "Age Group"
""")

┌───────────┬────────────────────────────────┐
│ Age Group │ count(DISTINCT patientAge.mrn) │
│  varchar  │             int64              │
├───────────┼────────────────────────────────┤
│ Senior    │                             73 │
│ Adult     │                            119 │
│ Pediatric │                             45 │
└───────────┴────────────────────────────────┘

## Analysis Example 3

   This is a count of female patients 40 years of age and older who have not previously had mammograms. This could be reviewed for marketing purposes.

In [12]:
duckdb.sql("""
    WITH prevMammo AS(
    SELECT
        v.mrn
    FROM charges as c
    JOIN 'Data Sets/Imaging Center Data/chgMast_cleaned_SQL.csv' as m
        ON c.chgNo = m.chgNo
    JOIN 'Data Sets/Imaging Center Data/visits_cleaned_SQL.csv' as v
        ON c.acctNum = v.acctNum
    WHERE m.revCode IN(0401, 0403)
    )
    SELECT
        COUNT(DISTINCT v.mrn) as "Females >= 40 W/Out Previous Mammo"
    FROM 'Data Sets/Imaging Center Data/charges_cleaned_SQL.csv' as c
    JOIN 'Data Sets/Imaging Center Data/chgMast_cleaned_SQL.csv' as m
        ON c.chgNo = m.chgNo
    JOIN 'Data Sets/Imaging Center Data/visits_cleaned_SQL.csv' as v
        ON c.acctNum = v.acctNum
    WHERE v.zip = 30547
    AND DATE_ADD(v.DOB, INTERVAL 40 YEAR) < current_date
""")

┌────────────────────────────────────┐
│ Females >= 40 W/Out Previous Mammo │
│               int64                │
├────────────────────────────────────┤
│                                142 │
└────────────────────────────────────┘

## Findings and Limitations

 - The cleaning process standardized fields, parsed mixed date formats, and exported analysis-ready files.
 - Validation checks were used to review missing keys and unmatched relationships.
 - The data itself is synthetic. However it is the same type of data used on a real world scenario created from a client request.